# 避難所候補算出支援ツール（sheltermatch）

要支援者一覧（緯度・経度入り）と避難所一覧の座標から、要支援者ごとに **距離が近い避難所候補（既定で上位3件）** を
算出し、必要であればハザード区域との位置関係も確認して、職員の最終判断のための資料（CSV）を作成するNotebookです。

**要支援者一覧CSVの正式フォーマット**: `resident_id,address,latitude,longitude,geocode_status` の
5列です（テンプレート: `templates/residents.csv`）。これは別Notebook `address_geocode.ipynb`
（住所→座標変換）の出力と同じ形式のため、`address_geocode.ipynb` で変換したCSVは、**加工せずそのまま
ここへアップロード**できます。

```text
templates/residents.csv
  ↓
address_geocode.ipynb（住所→座標変換）
  ↓
同じ5列のCSV
  ↓
sheltermatch.ipynb（このNotebook）
```

住所しかない場合は、先に `address_geocode.ipynb` で緯度・経度を付与してください（このNotebook自体は
住所→座標変換は行いません）。5列がすべて揃っていない要支援者CSVは受け付けません。

**このツールが行うこと**
- 直線距離（`geopy.distance.geodesic`）による避難所候補の算出（候補提示。自動割当ではありません）
- 結果を地図上で目視確認するためのレビュー用HTMLの作成（`sheltermatch_review.zip`。正式なデータ
  成果物は `assigned_shelters.csv` で、HTMLはその確認用の補助成果物です。背景地図（国土地理院の
  地理院タイルを画像化したもの）・ハザード区域PNG・地図ライブラリを同梱するため、閉域環境でも
  外部通信なしで開けます）
- 任意機能として、要支援者・避難所・両者を結ぶ直線とハザード区域（GeoJSON / Shapefile。国・県等の
  公式配布ZIPも展開・変換せずそのままアップロード可能）との位置関係の確認

**このツールが行わないこと（重要）**
- 住所→座標変換（要支援者一覧CSVの `latitude` / `longitude` にあらかじめ座標を入力してください。
  住所しかない場合は、別Notebookの `address_geocode.ipynb` で事前に緯度・経度を付与してから、
  その出力CSVをここへアップロードしてください）
- 避難先・避難経路の自動決定、道路経路・通行可能性の計算（直線距離であり道路距離ではありません。
  直線とハザード区域の交差判定も、道路上の避難経路判定ではありません）
- ハザード判定結果による候補避難所の自動除外・自動順位変更、独自の危険度スコアリング

距離順位とハザード判定は別々の情報として出力します。どの避難所を選ぶかは、出力結果を確認した **職員が判断** してください。

**要支援者CSVの座標が使えなかった場合**
- `match_status` = `no_coordinates`（`latitude` / `longitude` が空欄）
- `match_status` = `invalid_coordinates`（値は入っているが、数値として読めない・緯度経度の範囲外）
- どちらの場合も行は削除せず、入力した `latitude` / `longitude` もそのまま結果CSVへ残します
  （どの行をどう直せばよいか、結果CSVだけで分かるようにするためです）

**ハザード判定結果の読み方**
- `True` = ハザード区域内（境界上含む）または交差あり
- `False` = 有効な座標で判定した結果、ハザード区域外
- 空欄（NaN） = 座標が無い・不正、または候補自体が無いなどの理由で **判定できなかった**（区域外という意味ではありません）

**個人情報の取り扱い**
- 実際の要支援者データ・ハザードデータはこのリポジトリにコミットしないでください（サンプルを追加する場合は完全な架空データを使用してください）。
- 出力CSVには住所・座標等の個人情報が含まれ得ます。取り扱いに注意してください。

## 使い方

1. 下の「利用者設定」を確認する
2. 「ランタイム → すべてのセルを実行」
3. 要支援者CSV（`resident_id,address,latitude,longitude,geocode_status` の5列必須。
   `latitude` / `longitude` の値は行ごとに欠損・不正でも構いません。`resident_id` は空欄・重複不可です。
   `address_geocode.ipynb` の出力CSVはそのままアップロードできます）をアップロードする
4. 必要な場合のみハザードデータ（GeoJSON・Shapefile、または国・県等の公式配布ZIP）をアップロードする
5. BODIK Data APIからの避難所取得に失敗した場合のみ、避難所CSVをアップロードする
6. 結果CSV（`assigned_shelters.csv`）と、レビュー用HTML（`sheltermatch_review.zip`）を保存する
   （ZIPを展開して `review.html` を開くと、背景地図（国土地理院）の上に候補・ハザードを外部通信
   なしで確認できます。レビュー用HTML作成時のみ、背景地図・ハザードPNGの生成にインターネット
   接続が必要です）

コードを読み込まなくても、この説明と各セルのprint出力だけで操作できます。


In [ ]:
# ===== 利用者設定 =====
# 通常変更が必要な項目はこれだけです。値を確認・変更してから実行してください。

# 要支援者・避難所とハザード区域（GeoJSON）との位置関係を確認する場合は True にしてください。
ENABLE_HAZARD_CHECK = False

# 要支援者ごとに算出する避難所候補の件数（避難所がこの件数未満の場合は存在する件数まで出力）
TOP_N = 3

# 避難所一覧の取得方法。"api"=BODIK Data APIから取得 / "csv"=CSVファイルをアップロード
# "api"で取得に失敗した場合は、自動的にCSVアップロードへ切り替わります。
SHELTER_SOURCE = "api"

if not isinstance(TOP_N, int) or TOP_N < 1:
    raise ValueError(f"TOP_N は1以上の整数を指定してください。現在の値: {TOP_N!r}")

if SHELTER_SOURCE not in ("api", "csv"):
    raise ValueError(f"SHELTER_SOURCE は 'api' または 'csv' を指定してください。現在の値: {SHELTER_SOURCE!r}")

print("利用者設定を読み込みました。")
print(f"  ENABLE_HAZARD_CHECK = {ENABLE_HAZARD_CHECK}")
print(f"  TOP_N               = {TOP_N}")
print(f"  SHELTER_SOURCE      = '{SHELTER_SOURCE}'")


In [ ]:
# ===== 実行環境準備 =====
# Google Colabに標準で入っていないライブラリをインストールします（初回のみ数十秒かかることがあります）。
# matplotlib / pillow は、後でGitHubから読み込むレビュー生成モジュールが使います。
%pip install -q geopy geopandas shapely matplotlib pillow

import io
import re
import tempfile
import time
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from geopy.distance import geodesic

import geopandas as gpd
from shapely import make_valid, union_all
from shapely.geometry import Point, LineString

from google.colab import files

# 工程別の処理時間（秒）を記録する軽量な計測用の入れ物。time.perf_counter()による簡易計測のためだけに
# 追加したもので、既存の処理結果・ロジックは変更しない（詳細なprofilingは行わない）。
TIMINGS = {}

print("ライブラリの読み込みが完了しました。")


In [ ]:
# ===== 要支援者CSV読込・入力チェック =====
# 要支援者一覧CSVを選択してください（ファイル名は自由です）。共通住民CSVの正式フォーマットは
# resident_id,address,latitude,longitude,geocode_status の5列です（address_geocode.ipynbの
# 出力と同じ形式のため、そのままアップロードできます）。5列すべてを必須列として確認し、
# 不足していれば処理を止めます。resident_idは住所変換結果との対応が崩れないよう、空欄・重複が
# あればここでも処理を止めます（address_geocode.ipynbと同じ最低限のチェック）。
# latitude / longitude の値は行ごとに欠損・不正でも構いません（行を削除せず、後続の距離計算のみ
# 対象外とします。match_status 列で no_coordinates（空欄）/ invalid_coordinates（値は入って
# いるが数値として読めない・範囲外）として確認できます）。入力した値は書き換えず、そのまま
# 結果CSVへ残します。
# geocode_status列は変更せずそのまま結果CSVへ保持します（座標→避難所候補の突合結果である
# sheltermatch自身のmatch_status列とは別の情報のため、混同しないでください）。
# 文字コードは UTF-8 (BOM付き) → CP932 → UTF-8 の順に自動判定します。resident_idは
# address_geocode.ipynbと同様、先頭ゼロ等の表記を保つため文字列として読み込みます。

print("【要支援者一覧CSV】を選択してください。")
uploaded_residents = files.upload()

if len(uploaded_residents) == 0:
    raise RuntimeError("要支援者一覧CSVがアップロードされませんでした。ファイルを1つ選択してください。")
if len(uploaded_residents) > 1:
    raise RuntimeError("要支援者一覧CSVは1つだけ選択してください。")

residents_filename = list(uploaded_residents.keys())[0]
residents_bytes = uploaded_residents[residents_filename]
print(f"'{residents_filename}' を要支援者一覧として受け取りました。")

# 要支援者CSVを新たに読み込むたびに、この時点から性能計測をやり直す（TIMINGSをリセットする）。
# ハザードデータ・避難所データを読み込み直さずに、この「要支援者CSV読込・入力チェック」セルから
# 100/500/1000件を切り替えて再実行した場合に、前回実行分のhazard/shelters等の時間が今回の
# 合計へ混入しないようにするための措置（ハザードGeoDataFrame・避難所DataFrame自体は再利用できる。
# 計測用の辞書だけを空にする）。
TIMINGS = {}

_stage_start = time.perf_counter()


def read_csv_auto(file_bytes, label, dtype=None):
    """UTF-8(BOM付き) → CP932 → UTF-8 の順で読み込みを試み、成功したDataFrameを返す。"""
    encodings = [
        ("utf-8-sig", "UTF-8 (BOM付き)"),
        ("cp932", "CP932 (Shift-JIS系)"),
        ("utf-8", "UTF-8"),
    ]
    last_error = None
    for encoding, encoding_label in encodings:
        try:
            df = pd.read_csv(io.BytesIO(file_bytes), encoding=encoding, dtype=dtype)
            print(f"[{label}] {encoding_label} として読み込みました。（{len(df)}行）")
            return df
        except (UnicodeDecodeError, UnicodeError) as error:
            last_error = error
            continue
    raise ValueError(
        f"[{label}] 文字コードを判定できませんでした。"
        "UTF-8(BOM付き)・CP932・UTF-8のいずれでも読み込めません。"
        "Excel等での保存時の文字コードを確認してください。"
        f" 詳細: {last_error}"
    )


def ensure_coordinate_columns(df, label):
    """latitude/longitude列が両方存在することを確認する（値の欠損・不正は許容し、ここでは
    行を削除しない。列自体が無い場合のみ列不足として停止する）。避難所一覧CSV用。"""
    missing = [c for c in ("latitude", "longitude") if c not in df.columns]
    if missing:
        raise ValueError(f"[{label}] 必須列が見つかりません: {', '.join(missing)}")
    return df


def is_blank(series):
    """CSVの1列について、値が空欄（欠損、または空白だけ）の行をTrueとするSeriesを返す。
    「空欄」と「値は入っているが使えない」を区別するため、複数の入力チェックから共通で使う。"""
    return series.isna() | (series.astype(str).str.strip() == "")


# 共通住民CSVの正式フォーマット（address_geocode.ipynbの入出力と同じ5列）。
RESIDENT_REQUIRED_COLUMNS = ["resident_id", "address", "latitude", "longitude", "geocode_status"]


def ensure_resident_columns(df, label):
    """要支援者一覧CSVが共通住民CSVの5列（resident_id,address,latitude,longitude,geocode_status）
    をすべて持つことを確認する。列そのものは5列とも必須とし、不足していれば処理を止める
    （住所変換input＝output＝sheltermatch inputという共通フォーマットを一本化するため、
    旧来のlatitude/longitudeのみの形式は正式入力として扱わない）。latitude/longitudeの
    セルの値が欠損・不正な行は、従来どおり削除せず許容する。"""
    missing = [c for c in RESIDENT_REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(
            f"[{label}] 必須列が見つかりません: {', '.join(missing)}\n"
            f"共通住民CSVは {','.join(RESIDENT_REQUIRED_COLUMNS)} の5列です。"
            "templates/residents.csv を使用してください。"
        )
    return df


def find_resident_id_problems(df):
    """resident_id列の空欄・重複を検出する。resident_idは住所変換結果と避難所候補算出結果を
    紐づける結合キーのため、曖昧な状態のまま処理を進めない（address_geocode.ipynbと同じチェック）。
    戻り値は問題を説明する文字列のリスト（問題が無ければ空リスト）。行番号は職員が見るCSVの
    行番号（見出しを1行目として数えるため、0始まりの行位置に2を足した値）で示す。"""
    problems = []
    resident_id = df["resident_id"]
    blank_mask = is_blank(resident_id)
    duplicate_mask = resident_id.duplicated(keep=False) & ~blank_mask

    blank_rows = [position + 2 for position, blank in enumerate(blank_mask) if blank]
    if blank_rows:
        problems.append(f"resident_id が空欄の行があります（CSV行番号: {', '.join(map(str, blank_rows))}）")

    duplicate_detail = [
        f"CSV行{position + 2}='{value}'"
        for position, (value, duplicated) in enumerate(zip(resident_id, duplicate_mask))
        if duplicated
    ]
    if duplicate_detail:
        problems.append(f"resident_id が重複している行があります（{', '.join(duplicate_detail)}）")

    return problems


def is_valid_coordinate(lat, lon):
    """緯度・経度が数値として有効な範囲かどうかを返す（欠損・範囲外はFalse）。
    距離計算・ハザード判定など、1点ずつ座標を扱う関数から共通で利用する。"""
    if pd.isna(lat) or pd.isna(lon):
        return False
    return (-90 <= lat <= 90) and (-180 <= lon <= 180)


def read_coordinates(df, label):
    """latitude/longitude列を数値として読み取り、行ごとの状態を判定する。
    戻り値は (緯度Series, 経度Series, 状態Series) で、状態は次の3種類。

      ok                  : 距離計算に使える有効な座標
      no_coordinates      : 緯度・経度のどちらかが空欄
      invalid_coordinates : 値は入っているが、数値として読めない（例:「不明」）か、
                            緯度-90〜90・経度-180〜180の範囲外

    「空欄」と「値は入っているが使えない」を取り違えると、職員がどの行をどう直せばよいか
    分からなくなるため、必ず区別する。元のlatitude/longitude列は書き換えないので、
    入力した値はそのまま結果CSVに残る。"""
    lat = pd.to_numeric(df["latitude"], errors="coerce")
    lon = pd.to_numeric(df["longitude"], errors="coerce")

    status = pd.Series("ok", index=df.index)
    status[~(lat.between(-90, 90) & lon.between(-180, 180))] = "invalid_coordinates"
    status[is_blank(df["latitude"]) | is_blank(df["longitude"])] = "no_coordinates"

    unusable_count = int((status != "ok").sum())
    if unusable_count:
        print(f"[{label}] 距離計算に使えない座標の行が {unusable_count}件あります（全{len(df)}行中）。")
    return lat, lon, status


residents = read_csv_auto(residents_bytes, "要支援者一覧", dtype={"resident_id": str})
residents = ensure_resident_columns(residents, "要支援者一覧")

if len(residents) == 0:
    raise ValueError(
        "[要支援者一覧] データ行が1件もありません（見出し行だけのCSVです）。"
        "要支援者の行が入ったCSVをアップロードしてから、再度実行してください。"
    )

resident_id_problems = find_resident_id_problems(residents)
if resident_id_problems:
    raise ValueError(
        "[要支援者一覧] resident_id の内容に問題があるため処理を中止しました。\n"
        + "\n".join(f"  - {problem}" for problem in resident_id_problems)
        + "\nresident_id を空欄・重複のない一意な値に修正してから、再度アップロードしてください。"
    )

resident_lat, resident_lon, resident_coord_status = read_coordinates(residents, "要支援者一覧")

display(residents.head())

TIMINGS["residents_csv"] = time.perf_counter() - _stage_start
print(f"[処理時間] 要支援者CSV読込・入力チェック: {TIMINGS['residents_csv']:.2f}秒")


In [ ]:
# ===== 避難所取得・正規化・入力チェック =====
# 避難所一覧は上の「利用者設定」の SHELTER_SOURCE に従って取得します。
# - "api"（既定）: BODIK Data API（CKANの datastore_search）から自治体標準ODSの避難所データを
#   直接取得します（公開データの読み取りのみのためAPIキーは不要）。取得に失敗した場合
#   （通信エラー・レスポンス異常・0件など）は、Notebookを停止せずCSVアップロードへ自動的に切り替えます。
# - "csv": 避難所一覧CSVをブラウザから選択してアップロードします。
#
# 避難所一覧CSVが自治体標準オープンデータセット(ODS)形式（例:
# https://data.bodik.jp/dataset/472107_evacuation_space ）の場合、日本語列名
# （名称→name、緯度→latitude、経度→longitude）を自動的に内部標準列名へ変換します。
# 従来の name/latitude/longitude 形式のCSVもそのまま利用できます。
# 「災害種別_」で始まる列がある場合は、値の 1(対応済み)/2(2階以上であれば対応済み)/空欄(未対応) の
# 区別を保ったまま、避難所候補ごとにCSVへ参考情報として出力します（距離順位や候補の自動除外には
# 使用しません。これは指定緊急避難場所としての対応種別情報であり、後述のGeoJSONによるハザード判定
# とは別の情報です）。
# 座標が不正な避難所の行は距離計算の対象から除外します。

_stage_start = time.perf_counter()

BODIK_BASE_URL = "https://data.bodik.jp"
BODIK_RESOURCE_ID = "3132a0a4-f522-4b2d-bf18-f106d8b3a5ae"  # 糸満市 指定緊急避難場所データセット


def fetch_shelters_from_bodik(base_url, resource_id, page_size=1000):
    """BODIKのCKAN Data API(datastore_search)から避難所データを全件取得し、DataFrameで返す。
    total/offsetでページングして全件取得する。取得できない場合は例外を発生させ、
    呼び出し側でCSVアップロードへフォールバックする。"""
    endpoint = f"{base_url}/api/action/datastore_search"
    records = []
    offset = 0
    total = None

    while True:
        response = requests.get(
            endpoint,
            params={"resource_id": resource_id, "limit": page_size, "offset": offset},
            timeout=30,
        )
        response.raise_for_status()
        payload = response.json()

        if not payload.get("success"):
            raise RuntimeError("CKAN APIレスポンスが success=false を返しました。")

        result = payload.get("result")
        if result is None or "records" not in result:
            raise RuntimeError("CKAN APIレスポンスに result.records が含まれていません。")

        page_records = result["records"]
        records.extend(page_records)

        if total is None:
            total = result.get("total", len(page_records))

        offset += len(page_records)
        if len(page_records) == 0 or offset >= total:
            break

    if len(records) == 0:
        raise RuntimeError("BODIK APIの取得結果が0件でした。")

    return pd.DataFrame(records)


shelters_raw = None
shelters_bytes = None

if SHELTER_SOURCE == "api":
    try:
        shelters_raw = fetch_shelters_from_bodik(BODIK_BASE_URL, BODIK_RESOURCE_ID)
        print(f"BODIK APIから避難所一覧を{len(shelters_raw)}件取得しました。")
    except Exception as error:
        print("BODIK APIから避難所一覧を取得できませんでした。")
        print("CSVファイルから読み込みます。")
        print(f"（詳細: {error}）")

if shelters_raw is None:
    print("【避難所一覧CSV】を選択してください。")
    uploaded_shelters = files.upload()

    if len(uploaded_shelters) == 0:
        raise RuntimeError("避難所一覧CSVがアップロードされませんでした。ファイルを1つ選択してください。")
    if len(uploaded_shelters) > 1:
        raise RuntimeError("避難所一覧CSVは1つだけ選択してください。")

    shelters_filename = list(uploaded_shelters.keys())[0]
    shelters_bytes = uploaded_shelters[shelters_filename]
    print(f"'{shelters_filename}' を避難所一覧として受け取りました。")

if shelters_bytes is not None:
    shelters_raw = read_csv_auto(shelters_bytes, "避難所一覧")

# 避難所一覧の列名アイリアス（自治体標準ODS等の日本語列名 → 内部標準列名）
SHELTER_COLUMN_ALIASES = {
    "name": ["名称"],
    "latitude": ["緯度"],
    "longitude": ["経度"],
}

# 避難所の災害種別列の接頭辞（この接頭辞で始まる列を動的にすべて認識する）
DISASTER_TYPE_COLUMN_PREFIX = "災害種別_"

# 自治体標準ODSの災害種別列の値と対応区分の対応表（BODIKの指定緊急避難場所データセット仕様）
# 1=対応済み、2=2階以上であれば対応済み、空欄=未対応。これ以外の値は独自に「対応」と推測しない。
DISASTER_SUPPORT_LABELS = {
    "1": "対応済み",
    "2": "2階以上であれば対応済み",
}


def normalize_code_value(value):
    """公式データのコード値を、対応表を引くための文字列へ揃える。同じコードでも読込方法によって
    1 / "1" / 1.0 のように型や表現が変わるため、整数として解釈できる場合は整数の文字列表現
    （"1"）に統一する。欠損値はNoneを返す。1.5のような整数でない値を丸めると既知コードへ誤分類
    される恐れがあるため丸めず、元の値の文字列表現のまま返す（未知のコード値も消さずに残す）。
    避難所の災害種別列と、ハザードデータのコード属性（A33等）の両方から使う。"""
    if pd.isna(value):
        return None

    text = str(value).strip()
    try:
        number = float(text)
        if number.is_integer():
            return str(int(number))
    except (TypeError, ValueError):
        pass
    return text


def normalize_shelter_columns(df, label):
    """自治体標準ODS等で使われる日本語列名を、内部標準列名(name/latitude/longitude)へ変換する。
    既に標準列名がある場合はそちらを優先し、変換しない。使用した列名の対応表も返す。"""
    df = df.copy()
    used_columns = {}
    for standard_col, aliases in SHELTER_COLUMN_ALIASES.items():
        if standard_col in df.columns:
            used_columns[standard_col] = standard_col
            continue
        for alias in aliases:
            if alias in df.columns:
                df = df.rename(columns={alias: standard_col})
                used_columns[standard_col] = alias
                break

    renamed = {std: orig for std, orig in used_columns.items() if orig != std}
    if renamed:
        mapping_text = ", ".join(f"{orig}→{std}" for std, orig in renamed.items())
        print(f"[{label}] 自治体標準ODS等の日本語列名を自動変換しました: {mapping_text}")

    return df, used_columns


def detect_disaster_type_columns(df):
    """列名が '災害種別_' で始まる列を、対応する災害種別情報として動的に検出する。
    特定の災害種別名の一覧に固定せず、実データに存在する列をそのまま採用する。"""
    return [c for c in df.columns if c.startswith(DISASTER_TYPE_COLUMN_PREFIX)]


def classify_disaster_support(value):
    """自治体標準ODSの災害種別列の値を解釈し、対応区分のラベルを返す（未対応ならNone）。
    '1'=対応済み、'2'=2階以上であれば対応済み、空欄=未対応という標準仕様に従う。
    それ以外の想定外の値は独自に「対応している」と推測せず、値をそのまま保持して要確認として返す。"""
    code = normalize_code_value(value)
    if code is None or code == "":
        return None
    if code in DISASTER_SUPPORT_LABELS:
        return DISASTER_SUPPORT_LABELS[code]
    return f"{code}(要確認)"


def build_disaster_support(df, disaster_columns):
    """行ごとに対応している災害種別とその対応区分を ';' 区切りでまとめたSeriesを返す。
    例: '洪水:対応済み;高潮:2階以上であれば対応済み'。未対応(空欄)の種別は含めない。
    災害種別列が無ければ全行NaN。"""
    if not disaster_columns or len(df) == 0:
        return pd.Series([np.nan] * len(df), index=df.index, dtype="object")

    prefix_len = len(DISASTER_TYPE_COLUMN_PREFIX)

    def row_support(row):
        parts = []
        for col in disaster_columns:
            label = classify_disaster_support(row[col])
            if label is not None:
                parts.append(f"{col[prefix_len:]}:{label}")
        return ";".join(parts)

    return df.apply(row_support, axis=1)


# 避難所一覧は座標列の存否を確認する前に、自治体標準ODS等の日本語列名を内部標準列名へ変換する
shelters, shelter_used_columns = normalize_shelter_columns(shelters_raw, "避難所一覧")
shelter_disaster_columns = detect_disaster_type_columns(shelters)
shelters["_disaster_support"] = build_disaster_support(shelters, shelter_disaster_columns)
HAS_DISASTER_TYPE_COLUMNS = len(shelter_disaster_columns) > 0

shelters = ensure_coordinate_columns(shelters, "避難所一覧")
if "name" not in shelters.columns:
    raise ValueError("[避難所一覧] 名称列が見つかりません: name または 名称 の列が必要です。")

shelters["latitude"], shelters["longitude"], shelter_coord_status = read_coordinates(
    shelters, "避難所一覧"
)

# 座標が不正な避難所の行は距離計算の対象から除外する（有効な避難所が0件の場合は処理を停止する）
shelters_valid = shelters.loc[shelter_coord_status == "ok"].reset_index(drop=True)
invalid_shelter_count = len(shelters) - len(shelters_valid)

if len(shelters_valid) == 0:
    raise RuntimeError(
        "有効な座標を持つ避難所が0件です。避難所一覧CSVの latitude / longitude 列を確認してください。"
    )

print(
    f"[避難所一覧] 避難所件数: {len(shelters)}件 / "
    f"名称列: '{shelter_used_columns.get('name', 'name')}' / "
    f"緯度列: '{shelter_used_columns.get('latitude', 'latitude')}' / "
    f"経度列: '{shelter_used_columns.get('longitude', 'longitude')}' / "
    f"認識した災害種別列数: {len(shelter_disaster_columns)}件"
)
print(f"距離計算に使用する有効な避難所: {len(shelters_valid)}件（座標不正のため{invalid_shelter_count}件を除外）")

display(shelters.head())

TIMINGS["shelters"] = time.perf_counter() - _stage_start
print(f"[処理時間] 避難所データ取得・整形: {TIMINGS['shelters']:.2f}秒")


## ハザードデータ読込（任意）

`ENABLE_HAZARD_CHECK = True`（上の「利用者設定」）の場合のみ、ハザード区域のデータをアップロードします。
1ファイルにつき **`.geojson` 単体**、または **国・県等の公式配布ZIP（`.zip`）** のいずれかを選択できます。
ZIPの場合、内部のディレクトリ構造やファイル数に関わらず、配下の `.geojson` と **Shapefile**（`.shp`
とその組の `.shx`/`.dbf`。`.prj` は無くても構いません）を自動的に再帰探索してすべて統合します
（利用者が展開・変換・整理する必要はありません）。ZIP内のファイル名がCP932（Shift-JIS系）でUTF-8
フラグを立てずに格納されている場合（沖縄県公式データ等でよく見られます）も、自動的に文字化けを復元
してから展開します。`.shx`/`.dbf` が揃っていないShapefileは、そのレイヤーのみ読み込めない旨をまとめて
表示し、ほかに有効なレイヤーがあれば処理を継続します。区域判定に使えるPolygon/MultiPolygon以外の
ジオメトリ（LineString等）が含まれる場合は、独自に面へ変換したりはせず区域判定対象外として除外します。

公式配布データには、自己交差等で `is_valid=False` になっているPolygon/MultiPolygonが実際に含まれて
います（国土数値情報A31aと沖縄県津波浸水想定の実データで確認済み）。そのまま除外するとその区域が
判定から抜け落ちるため、**`is_valid=False` のPolygon/MultiPolygonに限り `shapely.make_valid()` で
直してから使います**。

- すでに有効なPolygon/MultiPolygonには `make_valid()` をかけず、一切変更しません
- 直した結果がGeometryCollection（面と線が混ざったもの）になる場合は、Polygon/MultiPolygon成分
  だけを使います（線・点の成分は区域ではないため使いません）
- 区域として使える形にならなかったものは、従来どおり除外します
- 空・欠損のジオメトリは、従来どおり区域判定対象外です
- 直した行も、元の `hazard_type`（A31aの詳細カテゴリ・A33の現象種類/区域区分・津波の浸水深区分・
  高潮のカテゴリ）をそのまま引き継ぎます。種別・カテゴリを再解釈することはありません

読込時には、不正なジオメトリが何件あり、そのうち何件を修復して採用し、何件を除外したのかを
まとめて表示します。

アップロードした **ファイル（GeoJSONまたはZIP）ごとに1回だけ** ハザード種別名を決定します（ZIP内の個々の
レイヤーごとには入力しません）。ファイル名が以下のような既知の公式配布形式に一致する場合は、入力を求めず
自動判定します（一致しない場合のみ、これまでどおり種別名の入力を求めます。空欄の場合はファイル名を
`hazard_type` とする挙動も維持されます）。

- 国土数値情報「洪水浸水想定区域データ」（`A31a-` で始まる形式。例: `A31a-25_47_10_GEOJSON.zip`）
  → 河川区分から「洪水（洪水予報河川・水位周知河川）」/「洪水（その他の河川）」を判定
- 国土数値情報「土砂災害警戒区域データ」（`A33-` で始まる形式。例: `A33-25_47_GEOJSON.zip`）→「土砂災害」
  （属性 `A33_001`/`A33_002` があれば現象の種類・区域区分も反映）
- 沖縄県津波浸水想定データ（`level1`〜`level7` で始まる形式。例: `level1_1cm-30cm.zip`）→「津波」
- 沖縄県高潮浸水想定データ（`<市町村コード>_takasiosinnsuisoutei_...` 形式。
  例: `47007_takasiosinnsuisoutei_22itoman.zip`）→「高潮」

ZIPの展開先直下にサブフォルダがある場合（国土数値情報等で「計画規模」「想定最大規模」のようにカテゴリ別に
ファイルが分かれている場合）は、そのサブフォルダ名を **カテゴリ** として扱い、`基本種別名:カテゴリ名` の形で
`hazard_type` に保持します。ただし高潮データは、複数のShapefile（`最大浸水深_糸満市.shp` /
`浸水継続時間_糸満市.shp` 等）が1つのラッパーフォルダにまとめて配布され、フォルダ名だけではレイヤーを
区別できないため、ラッパーフォルダの有無に関わらず、ファイル名から市町村名部分を除いた名前を優先して
カテゴリとします（例: `高潮:最大浸水深` / `高潮:浸水継続時間`）。また、属性に `分類` 列があるレイヤー
（津波浸水想定データ等）は、ポリゴンごとに `分類` の値をカテゴリとして反映します（例:
`津波:0.01m以上0.3m未満`）。土砂災害（A33）は `A33_001`（現象の種類）/`A33_002`（区域区分）の
公式コードリストの名称をそのまま用い、ポリゴンごとに `土砂災害:急傾斜地の崩壊:土砂災害警戒区域(指定済)`
のように反映します（未知のコード値が来た場合も削除せず、コード値自体を保持します）。フォルダ名・
ファイル名・属性の意味は独自解釈せずそのまま使うため、特定の配布元の命名規則以外には依存しません。

座標系はCRS情報がある場合はEPSG:4326へ変換します。CRS情報が無いGeoJSONは既定でWGS84として扱いますが、
CRS情報が無いShapefileについては無条件にEPSG:4326と決めつけず、座標値が経緯度として妥当な範囲
（経度-180〜180・緯度-90〜90）に収まる場合のみEPSG:4326とみなし、収まらない場合はそのレイヤーを座標系
不明として除外します。いずれの形式でもPolygon/MultiPolygon以外の空・不正なジオメトリは除外されます。
`ENABLE_HAZARD_CHECK = False`（既定）の場合はこのセルはスキップされ、ハザードデータなしで距離候補算出のみ
が実行されます。

**注意**: `ENABLE_HAZARD_CHECK = True` にした場合、有効なハザード区域ポリゴンが1件も読み込めなかったときは
「ハザードなし」とみなさず、ここで処理を停止します（判定していないことと、ハザード区域でないことを区別するためです）。


In [ ]:
# ===== ハザードデータ読込（任意） =====
# ENABLE_HAZARD_CHECK = True の場合のみ、ハザード区域のデータ（.geojson または公式配布ZIP）を
# アップロードして読み込みます。対応形式・ハザード種別の自動判定・ジオメトリの扱いは、
# 上の説明（markdownセル）を参照してください。


def _decode_zip_entry_name(member):
    """ZIPエントリのファイル名を復号する。UTF-8フラグ（汎用目的ビットフラグのbit 11）が
    立っていないエントリは、Pythonのzipfileが既定でCP437として解釈するため、CP932
    (Shift-JIS系)の日本語ファイル名を含む配布ZIP（沖縄県公式データ等、UTF-8フラグを立てずに
    作成されたZIP）では文字化けする。その場合は元のバイト列をCP932として再解釈する
    （変換できない場合は元の文字列のまま扱う）。"""
    if member.flag_bits & 0x800:
        return member.filename
    try:
        return member.filename.encode("cp437").decode("cp932")
    except (UnicodeDecodeError, UnicodeEncodeError):
        return member.filename


def _normalize_zip_entry_separators(name):
    """ZIPエントリ名のパス区切りを '/'（ZIP標準の区切り）に正規化する。国土数値情報A31a等、
    一部の公式配布ZIPはディレクトリ区切りに '\\'（バックスラッシュ）を使って格納されているが、
    '\\' はPOSIX環境のPathlibでは区切り文字として扱われずサブフォルダを認識できないため、
    展開前に '/' へ揃える。'/' はWindows・POSIXどちらのPathlibでも区切り文字として扱われるため、
    実行環境（Colab/Linux・Windows）に関わらず同じディレクトリ構造になる。"""
    return name.replace("\\", "/")


def safe_extract_zip(zip_bytes, extract_dir):
    """ZIPをextract_dirへ安全に展開する。各エントリ名は、文字コード復元
    （_decode_zip_entry_name）→パス区切り正規化（_normalize_zip_entry_separators）の順で
    処理してから扱う。絶対パスや'..'を含むなど、正規化後のパスが展開先ディレクトリの外に出る
    エントリが1件でもあれば、展開を一切行わずに例外を送出する（パストラバーサル対策。
    全エントリを先に検査してから展開する）。"""
    extract_dir_abs = Path(extract_dir).resolve()
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zip_file:
        resolved_members = []
        for member in zip_file.infolist():
            name = _decode_zip_entry_name(member)
            name = _normalize_zip_entry_separators(name)
            is_dir_entry = name.endswith("/")
            member_path = (extract_dir_abs / name).resolve()
            if member_path != extract_dir_abs and extract_dir_abs not in member_path.parents:
                raise RuntimeError(
                    "ZIP内に不正なパスが含まれているため展開を中止しました"
                    f"（パストラバーサルの可能性）: {name}"
                )
            resolved_members.append((member, member_path, is_dir_entry))

        for member, member_path, is_dir_entry in resolved_members:
            if is_dir_entry:
                member_path.mkdir(parents=True, exist_ok=True)
                continue
            member_path.parent.mkdir(parents=True, exist_ok=True)
            with zip_file.open(member) as source, open(member_path, "wb") as target:
                target.write(source.read())


def find_files_by_extension(extract_dir, extension):
    """extract_dir配下から、指定した拡張子のファイルを再帰的に探す。公式配布ZIPには
    '.SHP' / '.GEOJSON' のように拡張子が大文字で格納されているものがあり、Colab(Linux)の
    パターン照合は大文字・小文字を区別するため、ここでは区別せずに探す。"""
    return sorted(p for p in extract_dir.rglob("*") if p.is_file() and p.suffix.lower() == extension)


def find_shapefile_layers(extract_dir):
    """extract_dir配下の.shpを再帰的に探索し、同じディレクトリに.shx/.dbfが揃っているものだけを
    有効なShapefileレイヤーとして返す（.prjが無くても読み込みは試みる。CRSが無いものとして扱う）。
    戻り値は (有効な.shpパスのリスト, .shx/.dbfが揃っていないため除外した件数, 発見した.shp総数)。"""
    shp_paths = find_files_by_extension(extract_dir, ".shp")
    valid_paths = []
    missing_companion_count = 0
    for shp_path in shp_paths:
        siblings = {p.name.lower() for p in shp_path.parent.iterdir()}
        stem_lower = shp_path.stem.lower()
        if f"{stem_lower}.shx" in siblings and f"{stem_lower}.dbf" in siblings:
            valid_paths.append(shp_path)
        else:
            missing_companion_count += 1
    return valid_paths, missing_companion_count, len(shp_paths)


# 国土数値情報A33（土砂災害警戒区域データ）の公式コードリスト。
# A33_001=現象の種類、A33_002=区域区分。名称は公式コードリストのままとし、独自の呼称（イエロー/
# レッド等）へは置き換えない。
A33_PHENOMENON_LABELS = {
    "1": "急傾斜地の崩壊",
    "2": "土石流",
    "3": "地滑り",
}
A33_ZONE_LABELS = {
    "1": "土砂災害警戒区域(指定済)",
    "2": "土砂災害特別警戒区域(指定済)",
    "3": "土砂災害警戒区域(指定前)",
    "4": "土砂災害特別警戒区域(指定前)",
}


def derive_feature_hazard_types(gdf, hazard_type):
    """レイヤーの属性から、行（ポリゴン）ごとのhazard_typeを組み立てる。国土数値情報A33
    （土砂災害警戒区域データ）の 'A33_001'（現象の種類）/'A33_002'（区域区分）属性がある場合は
    公式コードリストの名称をそのまま用いて '<hazard_type>:<現象の種類>:<区域区分>' とする
    （未知のコード値は削除せず、コード値自体をそのまま使う）。津波浸水想定データ等の『分類』属性が
    ある場合は、従来どおり '<hazard_type>:<分類の値>' とする。どちらも無い場合はhazard_typeを
    全行へそのまま適用する。データセットが増えてもこの関数の中だけで判定し、呼び出し側を
    巨大なif文にしない。"""
    if "A33_001" in gdf.columns and "A33_002" in gdf.columns:
        def build_a33_hazard_type(row):
            phenomenon_code = normalize_code_value(row["A33_001"])
            zone_code = normalize_code_value(row["A33_002"])
            phenomenon = A33_PHENOMENON_LABELS.get(phenomenon_code, phenomenon_code)
            zone = A33_ZONE_LABELS.get(zone_code, zone_code)
            parts = [part for part in (phenomenon, zone) if part]
            return f"{hazard_type}:{':'.join(parts)}" if parts else hazard_type

        return gdf.apply(build_a33_hazard_type, axis=1)

    if "分類" in gdf.columns:
        return gdf["分類"].apply(
            lambda value: f"{hazard_type}:{value}" if pd.notna(value) and str(value).strip() else hazard_type
        )

    return hazard_type


# 区域判定に使えるジオメトリ種別。
POLYGON_TYPES = ["Polygon", "MultiPolygon"]


def empty_hazard_layer():
    """有効なハザード区域が1件も無い場合に返す、空のレイヤー。"""
    return gpd.GeoDataFrame({"hazard_type": [], "geometry": []}, crs="EPSG:4326")


def repair_invalid_polygon(geometry):
    """自己交差等で is_valid=False になっているPolygon/MultiPolygonを、shapely.make_valid()で
    区域判定に使える形へ直す。公式配布データには自己交差を含むポリゴンが実際に含まれており
    （国土数値情報A31a・沖縄県津波浸水想定で確認）、そのまま除外するとその区域が判定から
    抜け落ちるため、ここで直してから使う。

    make_valid()の結果がGeometryCollection（面と線が混ざったもの）になる場合は、
    Polygon/MultiPolygon成分だけを取り出す（線・点の成分は区域ではないため使わない）。
    区域として使える形にならなかった場合はNoneを返し、呼び出し側で従来どおり除外する。

    戻り値は (直したジオメトリ or None, GeometryCollectionから成分を取り出したか)。
    既に有効なジオメトリへは適用しないこと（この関数は呼び出し側でis_valid=Falseの行にだけ使う）。"""
    try:
        repaired = make_valid(geometry)
    except Exception:
        # make_valid自体が失敗した場合も、処理を止めずに従来どおり除外扱いにする
        return None, False

    if repaired is None or repaired.is_empty:
        return None, False

    from_collection = False
    if repaired.geom_type == "GeometryCollection":
        polygon_parts = [
            part for part in repaired.geoms
            if part.geom_type in POLYGON_TYPES and not part.is_empty
        ]
        if not polygon_parts:
            return None, False
        repaired = union_all(polygon_parts)
        from_collection = True

    if repaired.geom_type not in POLYGON_TYPES or repaired.is_empty or not repaired.is_valid:
        return None, False
    return repaired, from_collection


def load_hazard_layer(source, hazard_type, assume_wgs84_without_crs):
    """1件の空間データ（GeoJSONまたはShapefile）を読み込み、hazard_type/geometryの2列に正規化し、
    EPSG:4326へ統一する。区域判定はPolygon/MultiPolygonのみを対象とするため、次のように扱う。

    * 空・欠損のジオメトリ: 区域判定対象外として除外する
    * 有効なPolygon/MultiPolygon: 一切変更せずそのまま使う（make_valid()もかけない）
    * is_valid=FalseのPolygon/MultiPolygon: repair_invalid_polygonで直してから使う
      （直せなかったものは従来どおり除外する）
    * Polygon/MultiPolygon以外（LineString等）: 独自に面へ変換したりはせず除外する

    行ごとのhazard_typeはderive_feature_hazard_typesで組み立てる（『分類』属性やA33の公式属性が
    あれば詳細区分を反映し、無ければhazard_typeをそのまま使う）。ジオメトリを直した行も、
    元のhazard_typeをそのまま引き継ぐ（種別・カテゴリの再解釈はしない）。

    CRSが取得できる場合はEPSG:4326へ変換する。CRSが取得できない場合、assume_wgs84_without_crsが
    Trueなら（GeoJSON等、既定でWGS84として扱われる形式）そのままEPSG:4326とみなす。Falseの場合
    （Shapefile等）は無条件にEPSG:4326と決めつけず、geometry全体のboundsが経緯度として妥当な範囲
    （経度-180~180・緯度-90~90）に収まる場合のみEPSG:4326と推定し、収まらない場合は座標系を判断
    できないとみなして空のGeoDataFrameを返す（呼び出し側で除外扱いにする）。

    戻り値は (GeoDataFrame, crs_note, counts)。crs_noteは 'assumed_by_bounds' / 'unresolved' /
    None。countsは件数の内訳（null_or_empty / non_polygon / repaired /
    repaired_from_collection / unrepairable）で、利用者への集約表示に使う。"""
    counts = Counter()
    gdf = gpd.read_file(source)

    crs_note = None
    if gdf.crs is not None:
        if gdf.crs.to_epsg() != 4326:
            gdf = gdf.to_crs(epsg=4326)
    elif assume_wgs84_without_crs or len(gdf) == 0:
        gdf = gdf.set_crs(epsg=4326)
    else:
        minx, miny, maxx, maxy = gdf.total_bounds
        if -180 <= minx and maxx <= 180 and -90 <= miny and maxy <= 90:
            gdf = gdf.set_crs(epsg=4326)
            crs_note = "assumed_by_bounds"
        else:
            return empty_hazard_layer(), "unresolved", counts

    geometry = gdf.geometry
    is_null_or_empty = geometry.isna() | geometry.is_empty
    is_polygonal = geometry.geom_type.isin(POLYGON_TYPES)

    usable_mask = ((~is_null_or_empty) & is_polygonal & geometry.is_valid).to_numpy()
    repair_mask = ((~is_null_or_empty) & is_polygonal & (~geometry.is_valid)).to_numpy()

    counts["null_or_empty"] = int(is_null_or_empty.sum())
    counts["non_polygon"] = int(((~is_null_or_empty) & (~is_polygonal)).sum())

    # 有効なジオメトリはそのまま、is_valid=Falseのものだけ直して追加する。
    kept_positions = list(np.flatnonzero(usable_mask))
    kept_geometries = list(geometry.to_numpy()[usable_mask])

    for position in np.flatnonzero(repair_mask):
        repaired, from_collection = repair_invalid_polygon(geometry.iat[position])
        if repaired is None:
            counts["unrepairable"] += 1
            continue
        kept_positions.append(position)
        kept_geometries.append(repaired)
        counts["repaired"] += 1
        if from_collection:
            counts["repaired_from_collection"] += 1

    if not kept_positions:
        return empty_hazard_layer(), crs_note, counts

    kept_rows = gdf.iloc[kept_positions].reset_index(drop=True)
    layer = gpd.GeoDataFrame(
        {
            "hazard_type": derive_feature_hazard_types(kept_rows, hazard_type),
            "geometry": kept_geometries,
        },
        crs="EPSG:4326",
    )
    return layer, crs_note, counts


def _takashio_category_from_stem(stem):
    """高潮データ（例: 最大浸水深_糸満市.shp）のファイル名から、末尾の市町村名部分を除いた
    カテゴリ名を返す（例: '最大浸水深_糸満市' → '最大浸水深'）。区切りが無ければそのまま返す。"""
    prefix, sep, _rest = stem.rpartition("_")
    return prefix if sep else stem


def resolve_layer_hazard_type(base_hazard_type, vector_path, extract_dir):
    """ハザードのカテゴリをhazard_typeへ反映する。国土数値情報等では洪水の中でも「計画規模」
    「想定最大規模」等がZIP展開先直下のサブフォルダで分かれて配布されるため、そのサブフォルダ名を
    そのままカテゴリ名として使う（フォルダ名の意味は独自解釈せず、特定の配布元のディレクトリ名には
    依存しない）。ただし高潮（沖縄県高潮浸水想定データ）は複数のShapefileレイヤーが1つのラッパー
    フォルダにまとめて配布され、フォルダ名だけではレイヤーを区別できないため、フォルダの有無に
    関わらずファイル名から市町村名部分を除いたカテゴリ（例: 最大浸水深/浸水継続時間）を優先して
    用いる。サブフォルダも高潮の命名規則も無い場合はカテゴリなし（基本種別名のみ）とする。"""
    if base_hazard_type == "高潮":
        category = _takashio_category_from_stem(vector_path.stem)
    else:
        rel_parts = vector_path.relative_to(extract_dir).parts
        category = rel_parts[0] if len(rel_parts) > 1 else None
    return f"{base_hazard_type}:{category}" if category else base_hazard_type


def _detect_a31a_hazard_type(filename):
    """国土数値情報A31a（洪水浸水想定区域データ）のファイル名規則
    （例: A31a-25_47_10_GEOJSON.zip）から、年度・都道府県コードには依存せず、
    河川区分（10/20）のみで洪水の種別名を判定する。一致しなければNoneを返す。"""
    match = re.match(r"^A31a-\d+_\d+_(10|20)(?=[_.])", filename)
    if not match:
        return None
    river_classification_labels = {
        "10": "洪水（洪水予報河川・水位周知河川）",
        "20": "洪水（その他の河川）",
    }
    return river_classification_labels[match.group(1)]


def _detect_a33_hazard_type(filename):
    """国土数値情報A33（土砂災害警戒区域データ）のファイル名規則
    （例: A33-25_47_GEOJSON.zip）から、年度・都道府県コードには依存せず土砂災害と判定する。
    ファイル名の先頭部分だけを見るため、Colab等がファイル名重複を避けて末尾に付与する
    '(1)' 等の連番があっても判定できる。一致しなければNoneを返す。"""
    if re.match(r"^A33-\d+_\d+_GEOJSON", filename):
        return "土砂災害"
    return None


def _detect_tsunami_level_hazard_type(filename):
    """沖縄県津波浸水想定データのファイル名規則（例: level1_1cm-30cm.zip、level1～level7）から
    津波と判定する。浸水深の分類はファイル名のlevel番号からは推測せず、実データのShapefile内
    『分類』属性（load_hazard_layerで反映）を優先する。"""
    if re.match(r"^level[1-7](?=[_.])", filename, re.IGNORECASE):
        return "津波"
    return None


def _detect_takashio_hazard_type(filename):
    """沖縄県高潮浸水想定データのファイル名規則
    （例: 47007_takasiosinnsuisoutei_22itoman.zip）から高潮と判定する。"""
    if re.match(r"^\d+_takasiosinnsuisoutei_", filename):
        return "高潮"
    return None


# 既知の公式データ配布ファイル名パターンの判定関数一覧。今後、新しい公式データにも対応する場合は
# ここに判定関数を追加すればよく、巨大なif文へは積み上げない（現時点ではA31a・A33・
# 津波(level1~7)・高潮(takasiosinnsuisoutei)のみ、実データで確認できたファイル名規則として
# 実装している）。
KNOWN_HAZARD_FILENAME_DETECTORS = [
    _detect_a31a_hazard_type,
    _detect_a33_hazard_type,
    _detect_tsunami_level_hazard_type,
    _detect_takashio_hazard_type,
]


def detect_hazard_type(filename):
    """既知の公式データ配布ファイル名パターンからハザード種別名を自動判定する。ZIP・GeoJSON単体の
    どちらのファイル名にも同じ規則を適用する（形式ごとに判定ロジックを二重実装しない）。曖昧な推測は
    せず、既知のパターンのいずれにも一致しない場合はNoneを返す（呼び出し側で利用者入力へフォールバック
    する）。"""
    for detector in KNOWN_HAZARD_FILENAME_DETECTORS:
        hazard_type = detector(filename)
        if hazard_type is not None:
            return hazard_type
    return None


def load_hazard_upload(filename, file_bytes, hazard_type):
    """1つのアップロード（.geojson または国・県等の公式配布ZIP）から、GeoDataFrameを組み立てる。
    ZIPの場合は安全に展開し、内部のディレクトリ構造に関わらず配下の.geojsonと.shp（Shapefile。
    同名の.shx/.dbfが揃っているものに限る）を再帰的に探索する（特定の配布元のファイル名・
    ディレクトリ名には依存しない）。サブフォルダがあればカテゴリとしてhazard_typeに反映し、
    サブフォルダがなければ渡された基本種別名をそのままhazard_typeとする（高潮のみファイル名から
    カテゴリを補う。resolve_layer_hazard_type参照）。利用者には集約したサマリのみ表示し、
    ファイルごとの詳細ログは出さない。"""
    suffix = Path(filename).suffix.lower()
    if suffix not in (".geojson", ".zip"):
        raise ValueError(f"'{filename}' は対応していない形式です。.geojson または .zip を選択してください。")

    layers = []
    empty_file_count = 0
    unresolved_crs_count = 0
    assumed_by_bounds_count = 0
    missing_shapefile_companion_count = 0
    geometry_counts = Counter()

    if suffix == ".geojson":
        layer, crs_note, layer_counts = load_hazard_layer(
            io.BytesIO(file_bytes), hazard_type, assume_wgs84_without_crs=True
        )
        geometry_counts.update(layer_counts)
        if len(layer) == 0:
            empty_file_count += 1
        else:
            layers.append(layer)
    else:
        with tempfile.TemporaryDirectory(prefix="hazard_zip_") as extract_dir_str:
            extract_dir = Path(extract_dir_str)
            safe_extract_zip(file_bytes, extract_dir)
            print(f"'{filename}' を展開しました。")

            geojson_paths = find_files_by_extension(extract_dir, ".geojson")
            shp_paths, missing_shapefile_companion_count, total_shp_found = find_shapefile_layers(extract_dir)

            if not geojson_paths and total_shp_found == 0:
                raise RuntimeError(
                    f"'{filename}' 内に.geojsonまたはShapefile(.shp)が見つかりませんでした。"
                )

            print(f"Shapefileを {len(shp_paths)}レイヤー検出しました。")
            print(f"GeoJSONを {len(geojson_paths)}ファイル検出しました。")

            all_vector_paths = [(p, True) for p in geojson_paths] + [(p, False) for p in shp_paths]

            categories = sorted(
                {
                    p.relative_to(extract_dir).parts[0]
                    for p, _ in all_vector_paths
                    if len(p.relative_to(extract_dir).parts) > 1
                }
            )
            if categories:
                print(f"{len(categories)}カテゴリ（サブフォルダ）を検出しました。")

            for vector_path, is_geojson in all_vector_paths:
                file_hazard_type = resolve_layer_hazard_type(hazard_type, vector_path, extract_dir)
                layer, crs_note, layer_counts = load_hazard_layer(
                    vector_path, file_hazard_type, assume_wgs84_without_crs=is_geojson
                )
                geometry_counts.update(layer_counts)
                if crs_note == "unresolved":
                    unresolved_crs_count += 1
                    continue
                if crs_note == "assumed_by_bounds":
                    assumed_by_bounds_count += 1
                if len(layer) == 0:
                    empty_file_count += 1
                else:
                    layers.append(layer)

    if missing_shapefile_companion_count:
        print(
            f"{missing_shapefile_companion_count}件のShapefileは.shx/.dbfが揃っていないため"
            "読み込めませんでした。"
        )
    if assumed_by_bounds_count:
        print(
            f"{assumed_by_bounds_count}件はCRS情報が無いため、座標値から経緯度データと判断して"
            "EPSG:4326として読み込みました。"
        )
    invalid_polygon_count = geometry_counts["repaired"] + geometry_counts["unrepairable"]
    if invalid_polygon_count:
        print(f"不正なジオメトリ（自己交差等）: {invalid_polygon_count}件")
        print(f"  make_validで修復して採用: {geometry_counts['repaired']}件")
        if geometry_counts["repaired_from_collection"]:
            print(
                "    うちGeometryCollectionからPolygon成分を採用: "
                f"{geometry_counts['repaired_from_collection']}件"
            )
        print(f"  修復できず除外: {geometry_counts['unrepairable']}件")
    if geometry_counts["null_or_empty"]:
        print(
            f"{geometry_counts['null_or_empty']}件は空・欠損のジオメトリのため区域判定対象外です。"
        )
    if geometry_counts["non_polygon"]:
        print(
            f"{geometry_counts['non_polygon']}件はPolygon/MultiPolygon以外のため除外しました"
            "（この区域はハザード判定に含まれません）。"
        )
    if empty_file_count:
        print(f"{empty_file_count}ファイルは有効なPolygon/MultiPolygonを含まないため除外しました。")
    if unresolved_crs_count:
        print(
            f"{unresolved_crs_count}件は座標系を特定できないため除外しました"
            "（CRS情報が無く、座標値も経緯度として妥当な範囲ではありませんでした）。"
        )

    if not layers:
        raise RuntimeError(
            f"'{filename}' から有効なハザード区域(Polygon/MultiPolygon)を1件も読み込めませんでした。"
        )

    combined = gpd.GeoDataFrame(pd.concat(layers, ignore_index=True), crs="EPSG:4326")
    print(f"有効なハザードポリゴンを {len(combined)}件読み込みました。")
    combined_hazard_types = sorted(combined["hazard_type"].unique())
    if len(combined_hazard_types) == 1:
        print(f"hazard_type='{combined_hazard_types[0]}'")
    else:
        print(f"hazard_type: {', '.join(combined_hazard_types)}")
    return combined


hazard_gdf = None

if ENABLE_HAZARD_CHECK:
    print("ハザード区域のデータ（.geojson または国・県等の公式配布ZIP）をアップロードしてください（複数選択可）。")
    uploaded_hazards = files.upload()

    _stage_start = time.perf_counter()

    hazard_layers = []
    hazard_load_errors = []
    for hazard_filename, hazard_bytes in uploaded_hazards.items():
        detected_hazard_type = detect_hazard_type(hazard_filename)
        if detected_hazard_type is not None:
            hazard_type = detected_hazard_type
            print(f"'{hazard_filename}'")
            print(f"→ ハザード種別を自動判定しました: {hazard_type}")
        else:
            hazard_type = input(
                f"'{hazard_filename}' のハザード種別を自動判定できませんでした。\n"
                "ハザード種別名を入力してください（例: 洪水, 土砂災害, 津波, 高潮。"
                "ZIP内にサブフォルダがあれば、フォルダ名がカテゴリとして自動的に追加されます）: "
            ).strip()
            if not hazard_type:
                hazard_type = hazard_filename
        # 選択した複数ファイルのうち一部だけ読込失敗した状態のまま候補・ハザード判定へ進めると、
        # 判定結果のFalse（有効な座標で判定した結果、区域外）が「選択した全ハザードデータで
        # 判定済み」と誤認されかねない。そのため1ファイルの読込失敗（形式非対応・ZIP内に
        # geojson/shpが無い・有効なポリゴンが無い等）で即座に生のtracebackで止めはしないが、
        # ここでは握りつぶさず内容を集約し、全ファイルの検査後に1件でも失敗があれば処理を
        # 停止する（5種類全部を必須にはしないが、選択したファイルは全て正常に読めることを
        # 結果生成の条件にする。fail-closed）。
        try:
            layer = load_hazard_upload(hazard_filename, hazard_bytes, hazard_type)
        except (RuntimeError, ValueError) as error:
            hazard_load_errors.append((hazard_filename, str(error)))
            continue
        hazard_layers.append(layer)

    if hazard_load_errors:
        error_list = "\n".join(f"  - {name}: {reason}" for name, reason in hazard_load_errors)
        raise RuntimeError(
            "ハザードデータの一部を読み込めなかったため、判定結果が不完全になることを防ぐため"
            "処理を停止しました。\n"
            "\n読み込めなかったファイル:\n"
            f"{error_list}\n"
            "\nファイルを確認するか、今回判定に使用しないファイルは選択から外して、"
            "「ハザードデータ読込」セルを再実行してください。"
        )

    if hazard_layers:
        hazard_gdf = gpd.GeoDataFrame(pd.concat(hazard_layers, ignore_index=True), crs="EPSG:4326")

    if hazard_gdf is None or len(hazard_gdf) == 0:
        raise RuntimeError(
            "ENABLE_HAZARD_CHECK=True ですが、有効なハザード区域(Polygon/MultiPolygon)を1件も読み込めませんでした。"
            "GeoJSON/ZIPファイルが正しくアップロードされているか、ジオメトリ形式を確認してください。"
            "ハザード判定を行わない場合は、上の「利用者設定」で ENABLE_HAZARD_CHECK=False にしてください。"
        )

    print(f"ハザードデータを合計 {len(hazard_gdf)}件読み込みました。")

    TIMINGS["hazard"] = time.perf_counter() - _stage_start
    print(f"[処理時間] ハザードデータ読込・統合: {TIMINGS['hazard']:.2f}秒")
else:
    print("ENABLE_HAZARD_CHECK=False のため、ハザードデータの読込をスキップします。")
    TIMINGS["hazard"] = 0.0


In [ ]:
# ===== 距離計算・ハザード判定関数 =====
# 各要支援者について、有効な避難所すべてとの直線距離（geopy.distance.geodesic、メートル単位）を計算し、
# 近い順に TOP_N 件を候補として算出します（道路距離ではありません）。並び替えは丸める前の距離で行い、
# メートル単位への丸め（小数1桁）はCSVに出力する値を作成する際にのみ行います。距離が同一の場合でも
# 結果順が実行ごとにばらつかないよう、避難所名を用いて順序を安定させます。
#
# ハザード判定関数は、要支援者地点・候補避難所地点がハザード区域の内部または境界上にあるか、また
# 両地点を結ぶ直線がハザード区域と交差するかを判定します（直線交差は道路上の避難経路判定ではありません）。
# 座標が欠損・範囲外で判定できない場合は、区域外(False)と混同しないよう NaN を返します。
# ハザード区域は10万件規模になるため、1回の判定ごとに全ポリゴンを調べると要支援者数に比例して
# 待ち時間が延びます。GeoPandasの空間インデックス（hazard_area.sindex。1度作れば以降は再利用
# されます）で交差し得るポリゴンだけに絞ってから判定します（判定結果は絞り込みの有無で変わりません）。


def build_shelter_records(shelters_df):
    """避難所DataFrameから、距離計算に使う項目だけを取り出したタプルのリストを作る。
    要支援者1人ごとにDataFrameを1行ずつ取り出し直すと人数分だけ無駄に時間がかかるため、
    最初に1回だけ作って全員で使い回す。名前は並び順を安定させるため文字列に揃える。"""
    has_disaster_info = "_disaster_support" in shelters_df.columns
    return [
        (
            str(shelter["name"]),
            shelter["latitude"],
            shelter["longitude"],
            shelter["_disaster_support"] if has_disaster_info else np.nan,
        )
        for _, shelter in shelters_df.iterrows()
    ]


def compute_candidates(resident_lat, resident_lon, shelter_records, top_n):
    """要支援者の座標から近い順に避難所候補を [(名前, 距離m, 緯度, 経度, 災害種別対応区分), ...] で返す。
    座標が欠損、または緯度・経度が有効範囲外であれば空リストを返す（geodesic()に不正値を渡さない）。
    距離は丸めずに返す（並び替え後、出力時にのみ丸める）。距離が同一の場合は避難所名で順序を
    安定させる。対応区分は災害種別列が無ければNaNのまま。"""
    if not is_valid_coordinate(resident_lat, resident_lon):
        return []

    resident_coord = (resident_lat, resident_lon)
    candidates = [
        (name, geodesic(resident_coord, (lat, lon)).meters, lat, lon, disaster_support)
        for name, lat, lon, disaster_support in shelter_records
    ]
    candidates.sort(key=lambda candidate: (candidate[1], candidate[0]))
    return candidates[:top_n]


def hazard_types_intersecting(geometry, hazard_area):
    """ジオメトリ（点または直線）と交差するハザード区域を調べ、
    (該当したか, 該当したhazard_typeを;で連結した文字列) を返す。"""
    hit_rows = hazard_area.sindex.query(geometry, predicate="intersects")
    hit_types = sorted(hazard_area["hazard_type"].iloc[hit_rows].unique())
    return (len(hit_types) > 0), ";".join(hit_types)


def hazard_types_at_point(lat, lon, hazard_area):
    """座標がハザード区域の内部または境界上にあるかどうかと、該当するhazard_type（;区切り）を返す。
    座標が欠損・範囲外の場合は判定不能としてnp.nanを返す（区域外=Falseと混同しない）。"""
    if not is_valid_coordinate(lat, lon):
        return np.nan, np.nan
    return hazard_types_intersecting(Point(lon, lat), hazard_area)


def hazard_types_on_line(lat1, lon1, lat2, lon2, hazard_area):
    """2地点を結ぶ直線がハザード区域と交差するかどうかと、該当するhazard_typeを返す（道路経路上の判定ではない）。
    いずれかの座標が欠損・範囲外の場合は判定不能としてnp.nanを返す。"""
    if not is_valid_coordinate(lat1, lon1) or not is_valid_coordinate(lat2, lon2):
        return np.nan, np.nan
    return hazard_types_intersecting(LineString([(lon1, lat1), (lon2, lat2)]), hazard_area)


print("距離計算・ハザード判定関数を定義しました。")


In [ ]:
# ===== 候補算出とハザード判定の実行 =====
# 要支援者ごとに、避難所候補・距離・（ENABLE_HAZARD_CHECK=True の場合のみ）ハザード判定結果を組み立てます。
# 距離による候補順位はハザード判定結果によって変更されません。座標がない・不正な要支援者の行も削除せず、
# 候補・距離を空欄のまま保持します（match_status 列で ok / no_coordinates(座標欄が空欄) /
# invalid_coordinates(値は入っているが数値として読めない・範囲外) の状態を確認できます）。
# ハザードデータを読み込んでいない状態でこのセルを実行した場合は、全行が「区域外」と誤読される結果を
# 出さないよう、ここで処理を止めます。
# ハザード関連列は、有効な座標で実際に判定できた場合のみ True/False とし、要支援者座標が欠損・不正な場合や
# 候補自体が存在しない場合は NaN（判定不能）のまま出力します（候補避難所地点は常に有効座標を持つため、
# 候補が存在すれば通常どおり True/False で判定します）。
# 避難所一覧に災害種別列があった場合は candidate_N_disaster_support 列に、災害種別ごとの対応区分
# （例: 洪水:対応済み;高潮:2階以上であれば対応済み）を出力します（対応区分によって候補の順位変更・自動除外は行いません）。

# ハザード判定を行う設定なのにハザードデータが読み込まれていない場合は、ここで処理を止める。
# このまま進めると全行が False（＝有効な座標で判定した結果、区域外）として出力され、
# 「判定していない」ことと「ハザード区域ではない」ことの区別が付かなくなるため。
if ENABLE_HAZARD_CHECK and (hazard_gdf is None or len(hazard_gdf) == 0):
    raise RuntimeError(
        "ENABLE_HAZARD_CHECK=True ですが、ハザードデータが読み込まれていません。"
        "先に「ハザードデータ読込」セルを実行してください"
        "（ハザード判定を行わない場合は、上の「利用者設定」で ENABLE_HAZARD_CHECK=False にしてください）。"
    )

_stage_start = time.perf_counter()

# 避難所の一覧は全員分の距離計算で共通のため、1回だけ作って使い回す。
shelter_records = build_shelter_records(shelters_valid)

# 候補避難所は要支援者ごとに同じ避難所がくり返し選ばれるため、避難所地点のハザード判定は
# 避難所ごとに1回だけ行い、結果を使い回す（同じ判定を人数分くり返さない）。
shelter_hazard_results = {}


def shelter_hazard_at(shelter_lat, shelter_lon):
    key = (shelter_lat, shelter_lon)
    if key not in shelter_hazard_results:
        shelter_hazard_results[key] = hazard_types_at_point(shelter_lat, shelter_lon, hazard_gdf)
    return shelter_hazard_results[key]


result_records = []

# レビュー用HTMLの表示データ。CSVと候補順位がずれないよう、候補はこのループで得たものを
# そのまま使う（HTML用に別の候補順位計算を持たない）。避難所の緯度・経度はCSVへは追加せず、
# 地図表示のためにここだけで保持する。
review_rows = []

for lat, lon, coord_status in zip(resident_lat, resident_lon, resident_coord_status):
    candidates = compute_candidates(lat, lon, shelter_records, TOP_N)
    review_candidates = []

    # match_statusは、職員が1行を横方向に追ったときに最初に目に入るよう先頭へ置く
    # （候補・距離を見てから「実は座標が無効だった」と分かる並びにしない）。
    record = {"match_status": coord_status}

    if ENABLE_HAZARD_CHECK:
        record["resident_in_hazard"], record["resident_hazard_types"] = hazard_types_at_point(
            lat, lon, hazard_gdf
        )

    for n in range(1, TOP_N + 1):
        # 候補が無い（座標が無効、または避難所件数がTOP_Nに満たない）場合は、判定できなかった
        # ことを示すため全項目を空欄（NaN）のままにする。
        shelter_name = distance_m = disaster_support = np.nan
        shelter_in_hazard = shelter_hazard_types = np.nan
        line_intersects = line_hazard_types = np.nan

        if n <= len(candidates):
            shelter_name, distance_raw, shelter_lat, shelter_lon, disaster_support = candidates[n - 1]
            distance_m = round(distance_raw, 1)
            if ENABLE_HAZARD_CHECK:
                shelter_in_hazard, shelter_hazard_types = shelter_hazard_at(shelter_lat, shelter_lon)
                line_intersects, line_hazard_types = hazard_types_on_line(
                    lat, lon, shelter_lat, shelter_lon, hazard_gdf
                )
            review_candidates.append({
                "rank": n,
                "name": shelter_name,
                "latitude": shelter_lat,
                "longitude": shelter_lon,
                "distance_m": distance_m,
                "disaster_support": disaster_support if HAS_DISASTER_TYPE_COLUMNS else None,
                "shelter_in_hazard": shelter_in_hazard,
                "shelter_hazard_types": shelter_hazard_types,
                "straight_line_intersects_hazard": line_intersects,
                "straight_line_hazard_types": line_hazard_types,
            })

        record[f"candidate_{n}"] = shelter_name
        record[f"distance_{n}_m"] = distance_m
        if HAS_DISASTER_TYPE_COLUMNS:
            record[f"candidate_{n}_disaster_support"] = disaster_support
        if ENABLE_HAZARD_CHECK:
            record[f"candidate_{n}_shelter_in_hazard"] = shelter_in_hazard
            record[f"candidate_{n}_shelter_hazard_types"] = shelter_hazard_types
            record[f"candidate_{n}_straight_line_intersects_hazard"] = line_intersects
            record[f"candidate_{n}_straight_line_hazard_types"] = line_hazard_types

    result_records.append(record)
    review_rows.append({
        "latitude": lat if coord_status == "ok" else None,
        "longitude": lon if coord_status == "ok" else None,
        "candidates": review_candidates,
    })

results_df = pd.DataFrame(result_records)

# 前回の結果CSV（assigned_shelters.csv）をそのまま再投入された場合、入力側にも同じ名前の列が
# 残っている。そのまま連結すると同名の列が二重になり、以降の集計がエラーで落ちるため、
# このNotebookが作る列は入力側から取り除いてから連結する。
#
# 取り除く対象は、下の一覧に挙げた「このNotebookが作る列名そのもの」に限る。
# candidate_1_備考 のように職員が独自に追加した列は、名前が似ていても取り除かない
# （入力されたデータを黙って消さないため）。
GENERATED_COLUMNS = ("match_status", "resident_in_hazard", "resident_hazard_types")

# 候補ごとに作る列。候補番号(N)は、前回と異なる TOP_N で再実行される場合があるため、
# 今回のTOP_Nに限定せず数字として判定する。
GENERATED_CANDIDATE_COLUMNS = (
    "candidate_{n}",
    "distance_{n}_m",
    "candidate_{n}_disaster_support",
    "candidate_{n}_shelter_in_hazard",
    "candidate_{n}_shelter_hazard_types",
    "candidate_{n}_straight_line_intersects_hazard",
    "candidate_{n}_straight_line_hazard_types",
)


def is_generated_column(column_name):
    """このNotebookが結果として作る列名かどうかを返す（上の一覧と完全に一致する名前のみTrue）。"""
    if column_name in GENERATED_COLUMNS:
        return True
    return any(
        re.fullmatch(template.format(n=r"\d+"), column_name)
        for template in GENERATED_CANDIDATE_COLUMNS
    )


reused_columns = [c for c in residents.columns if is_generated_column(c)]
if reused_columns:
    print(f"入力CSVに前回の結果列が含まれていたため、今回の結果で置き換えます: {', '.join(reused_columns)}")

final_df = pd.concat(
    [residents.drop(columns=reused_columns).reset_index(drop=True), results_df], axis=1
)

TIMINGS["candidates"] = time.perf_counter() - _stage_start
print("候補算出が完了しました。")
print(f"[処理時間] 避難所候補算出＋ハザード判定: {TIMINGS['candidates']:.2f}秒")


In [ ]:
# ===== レビュー生成モジュール準備 =====
# レビュー用HTML（補助成果物）の実装は、Notebookの外の通常のPython／HTMLファイルとして
# GitHubで保守しています。ここではそれを取得して読み込むだけです。
#
#   src/review/review_builder.py    … 地図素材の準備・ハザードPNG生成・データ組み立て・ZIP出力
#   src/review/review_template.html … review.html の画面（HTML・CSS・JavaScript）
#
# 外部（GitHub）へ通信するのは、このセルでレビュー生成の道具を取りに行くときだけです。
# できあがった sheltermatch_review.zip は、従来どおり外部通信なしに利用できます
# （地図ライブラリ・背景地図PNG・ハザードPNGはすべてZIPへ同梱されます）。

import importlib.util

REVIEW_BASE_URL = (
    "https://raw.githubusercontent.com/YanTKYS/sheltermatch/main/src/review"
)
# このNotebookが想定しているレビュー生成モジュールのインターフェース。取得したモジュールの
# REVIEW_BUILDER_API_VERSION と一致しない場合は、互換性のない組み合わせのまま進めずに止める。
EXPECTED_REVIEW_BUILDER_API_VERSION = 1

review_runtime_dir = Path("sheltermatch_review_runtime")
review_runtime_dir.mkdir(parents=True, exist_ok=True)

for review_filename in ("review_builder.py", "review_template.html"):
    try:
        review_response = requests.get(f"{REVIEW_BASE_URL}/{review_filename}", timeout=30)
        review_response.raise_for_status()
    except Exception as error:
        raise RuntimeError(
            "レビュー生成モジュールをGitHubから取得できませんでした。\n"
            "インターネット接続を確認して再実行してください。\n"
            f"（取得先: {REVIEW_BASE_URL}/{review_filename} / 詳細: {error}）"
        ) from error
    (review_runtime_dir / review_filename).write_bytes(review_response.content)

REVIEW_TEMPLATE_PATH = review_runtime_dir / "review_template.html"

# 取得したPythonコードを exec() で直接実行はせず、ファイルへ保存してからモジュールとして読み込む。
review_builder_spec = importlib.util.spec_from_file_location(
    "review_builder", review_runtime_dir / "review_builder.py"
)
review_builder = importlib.util.module_from_spec(review_builder_spec)
review_builder_spec.loader.exec_module(review_builder)

if review_builder.REVIEW_BUILDER_API_VERSION != EXPECTED_REVIEW_BUILDER_API_VERSION:
    raise RuntimeError(
        "レビュー生成モジュールの互換性を確認できません。\n"
        f"このNotebookが想定しているバージョン: {EXPECTED_REVIEW_BUILDER_API_VERSION} / "
        f"取得したモジュールのバージョン: {review_builder.REVIEW_BUILDER_API_VERSION}\n"
        "Notebookとレビュー生成モジュールの組み合わせを確認してください"
        "（GitHubの main から最新のNotebookを取得し直すと解消することがあります）。"
    )

print(f"レビュー生成モジュールを読み込みました（API version {review_builder.REVIEW_BUILDER_API_VERSION}）。")

In [ ]:
# ===== 結果確認・CSV出力・レビュー用HTML出力 =====
# CSVを出力する前に、Notebook上で処理結果の概要と先頭数行を確認します。
# 結果は Excelで文字化けしにくい utf-8-sig（UTF-8 BOM付き）でCSVに出力し、ブラウザへダウンロードします。
#
# 続けて、結果を地図上で目視確認するためのレビュー用HTMLを sheltermatch_review.zip として
# 出力します。ZIPを展開して review.html を開くと、要支援者を1人ずつ選んで候補・距離・ハザードを
# 地図で確認できます。正式なデータ成果物は assigned_shelters.csv で、HTMLはその確認用の
# 補助成果物です（避難所の割り当て結果ではありません）。
#
# 出力CSV・HTMLには個人情報が含まれ得るため、取り扱いに注意してください。

_stage_start = time.perf_counter()

total_residents = len(final_df)
ok_count = int((final_df["match_status"] == "ok").sum())
no_coord_count = int((final_df["match_status"] == "no_coordinates").sum())
invalid_coord_count = int((final_df["match_status"] == "invalid_coordinates").sum())

print(f"要支援者件数: {total_residents}件")
print(f"距離計算できた件数: {ok_count}件")
print(f"座標が空欄のため距離計算できなかった件数: {no_coord_count}件")
print(f"座標が範囲外で不正なため距離計算できなかった件数: {invalid_coord_count}件")
print(f"距離計算に使用した有効な避難所件数: {len(shelters_valid)}件")

if ENABLE_HAZARD_CHECK:
    def count_true(column_name_format):
        """候補1〜TOP_Nの指定列でTrueだった件数を合計する（延べ件数）。"""
        return sum(
            int(final_df[column_name_format.format(n=n)].eq(True).sum())
            for n in range(1, TOP_N + 1)
        )

    resident_hazard_count = int(final_df["resident_in_hazard"].eq(True).sum())
    shelter_hazard_count = count_true("candidate_{n}_shelter_in_hazard")
    line_hazard_count = count_true("candidate_{n}_straight_line_intersects_hazard")

    print(f"ハザード区域内（境界上含む）にいる要支援者数: {resident_hazard_count}件")
    print(f"ハザード区域内にある候補避難所の件数（延べ、TOP_N分の合計）: {shelter_hazard_count}件")
    print(f"候補避難所への直線がハザード区域と交差する件数（延べ、TOP_N分の合計）: {line_hazard_count}件")

display(final_df.head())

OUTPUT_FILENAME = "assigned_shelters.csv"

final_df.to_csv(OUTPUT_FILENAME, index=False, encoding="utf-8-sig")
print(f"'{OUTPUT_FILENAME}' を出力しました。")

# 正式なデータ成果物である結果CSVは、レビュー用HTMLの作成結果に関わらず受け取れるよう
# 先にダウンロードする。
files.download(OUTPUT_FILENAME)

# レビュー用HTMLは結果CSVと同じ実行結果から作る（CSVを読み直して独自解釈はしない）。
# 内容がCSVと食い違う場合は、build_review_package の中で処理を止める。
review_zip_path = review_builder.build_review_package(
    final_df=final_df,
    review_rows=review_rows,
    hazard_area=hazard_gdf,
    shelters_df=shelters_valid,
    top_n=TOP_N,
    output_dir=Path("."),
    template_path=REVIEW_TEMPLATE_PATH,
)
print(f"'{review_zip_path.name}' を出力しました（展開して review.html を開いてください）。")

TIMINGS["output"] = time.perf_counter() - _stage_start
print(f"[処理時間] 結果CSV・レビューHTML生成: {TIMINGS['output']:.2f}秒")

total_time = sum(TIMINGS.values())
print("\n処理時間:")
print(f"  要支援者CSV読込        {TIMINGS.get('residents_csv', 0):.1f}秒")
print(f"  避難所データ準備        {TIMINGS.get('shelters', 0):.1f}秒")
print(f"  ハザードデータ読込      {TIMINGS.get('hazard', 0):.1f}秒")
print(f"  候補・ハザード判定      {TIMINGS.get('candidates', 0):.1f}秒")
print(f"  CSV・HTML出力          {TIMINGS.get('output', 0):.1f}秒")
print(f"  合計                   {total_time:.1f}秒")

files.download(str(review_zip_path))
